In [1]:
import sys
sys.path.append("../../")

%load_ext autoreload
%autoreload 2

In [2]:
import optuna
import pickle
from functools import partial
from pathlib import Path

from simulator.simulation.modules import Campaign
from simulator.simulation.utils_visualization import data_prep_vis, plot_history_article
from simulator.simulation.simulate import simulate_campaign
from simulator.validation.check_results import autobidder_check

/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd

In [4]:
from simulator.model.rlb_dp_bidder import RLBDPBidder

In [5]:
auction_mode = "FPA"  # or "VCG"
best_params_subfolder = f"{auction_mode.lower()}_rlb_n10_rndm_42"
best_models_subfolder = f"{auction_mode.lower()}_rlb_n10_rndm_42"

# metric to optimize: CPC_REL / RMSE / SCR
metric = "SCR"
n_trials = 25

In [6]:
data_config = {
    "train": {
        "campaigns_path": f"../../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_train_final.csv",
        "stats_path": f"../../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_train_final.csv",
    },
    "test": {
        "campaigns_path": f"../../data/{auction_mode.lower()}/campaigns_{auction_mode.lower()}_filtered_test_final.csv",
        "stats_path": f"../../data/{auction_mode.lower()}/stats_{auction_mode.lower()}_filtered_test_final.csv",
    },
}

data_config

{'train': {'campaigns_path': '../../data/fpa/campaigns_fpa_filtered_train_final.csv',
  'stats_path': '../../data/fpa/stats_fpa_filtered_train_final.csv'},
 'test': {'campaigns_path': '../../data/fpa/campaigns_fpa_filtered_test_final.csv',
  'stats_path': '../../data/fpa/stats_fpa_filtered_test_final.csv'}}

In [7]:
stats_path = data_config['train']['stats_path']
campaigns_path = data_config['train']['campaigns_path']

In [8]:
stats_df = pd.read_csv(stats_path)

In [9]:
def objective_rlb_dp(trial, metric='RMSE_T', auction_mode='FPA'):

    max_bid = trial.suggest_float('max_bid', 10, 500, log=True)
    gamma = trial.suggest_float('gamma', 0.80, 1.00) 
    N_bound = trial.suggest_int('N_bound', 6, 72)
    B_bound = trial.suggest_int('B_bound', 1e3, 2e4, log=True)

    custom_params = {
        "max_bid": max_bid,
        "lower_clip": 5,
        "upper_clip": 5,
        "gamma": gamma,
        "model_path": None,
        "N_bound": N_bound,
        "B_bound": B_bound,
    }


    bidder = RLBDPBidder(custom_params)

    bidder.fit(stats_df)
    import os, uuid

    os.makedirs("tmp_models", exist_ok=True)
    TMP_MODEL_PATH = f"tmp_models/rlb_dp_trial{trial.number}_{uuid.uuid4().hex}.pkl"
    bidder.save_model(TMP_MODEL_PATH)

    # прогоняем через тот же пайплайн проверки
    res = autobidder_check(
        bidder=RLBDPBidder,
        params={
            "input_campaigns": campaigns_path,
            "input_stats": stats_path,
            "max_bid": max_bid,
            "gamma": gamma,
            "model_path": TMP_MODEL_PATH,
            "N_bound": N_bound,
            "B_bound": B_bound,
        },
        auction_mode=auction_mode,
    )

    print(f"CPC_REL: {res['score'][0]}, rmse: {res['score'][1]}, SCR: {res['score'][2]}")
    if metric == 'RMSE_T':
        return res['score'][1]
    elif metric == 'CPC_REL':
        return res['score'][0]
    elif metric == 'SCR':
        return res['score'][2]


def opt_search_rlb_dp(n_trials, metric='RMSE_T', auction_mode='FPA'):
    study = optuna.create_study(
        direction='maximize' if metric == 'SCR' else 'minimize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    
    study.optimize(
        partial(objective_rlb_dp, metric=metric, auction_mode=auction_mode),
        n_trials=n_trials,
        n_jobs=6
    )

    print('Best trial:')
    trial = study.best_trial
    print(f'  Value: {trial.value}')
    print('  Params: ')

    dict_path = f'best_params/rlb_dp_{metric.lower()}_{auction_mode}.pkl'
    params_dict = {}
    for key, value in trial.params.items():
        print(f'    {key}: {value}')
        params_dict[key] = value

    with open(dict_path, 'wb') as f:
        pickle.dump(params_dict, f)

    return study


def train_best_rlb_dp(best_params_path, model_path='rlb_dp_model_tuned.pkl'):
    """Обучить и сохранить модель с лучшими параметрами (после optuna)."""
    with open(best_params_path, 'rb') as f:
        best_params = pickle.load(f)

    custom_params = {
        "max_bid": best_params["max_bid"],
        "gamma": best_params["gamma"],
        "model_path": None,
        "N_bound": best_params["N_bound"],
        "B_bound": best_params["B_bound"],
    }

    bidder = RLBDPBidder(custom_params)
    bidder.fit(stats_df)
    bidder.save_model(model_path)
    return bidder


In [10]:
study_rlb = opt_search_rlb_dp(n_trials, metric, auction_mode)

[I 2026-04-28 11:51:57,888] A new study created in memory with name: no-name-20e85027-f688-4941-bcb4-89e5e7b266e3


Hours: 100%|██████████| 15/15 [00:00<00:00, 287.06it/s]






Hours:   0%|          | 0/21 [00:00<?, ?it/s]





Hours:  43%|████▎     | 9/21 [00:00<00:00, 84.10it/s]





Hours:  86%|████████▌ | 18/21 [00:00<00:00, 35.84it/s]




Hours: 100%|██████████| 13/13 [00:01<00:00,  9.16it/s]



Hours: 100%|██████████| 21/21 [00:00<00:00, 22.85it/s]



































Hours: 100%|██████████| 29/29 [00:02<00:00, 11.21it/s]










Hours: 100%|██████████| 38/38 [00:02<00:00, 13.98it/s]









Hours: 100%|██████████| 46/46 [00:02<00:00, 18.38it/s]
[I 2026-04-28 12:00:17,300] Trial 3 finished with value: 7392.788105074793 and parameters: {'max_bid': 284.04300429392185, 'gamma': 0.8302533682876002, 'N_bound': 38, 'B_bound': 6549}. Best is trial 3 with value: 7392.788105074793.


CPC_REL: 893.3080305894539, rmse: 2.676490515078634, SCR: 7392.788105074793


Hours: 100%|██████████| 50/50 [00:02<00:00, 22.71it/s]
[I 2026-04-28 12:01:57,990] Trial 0 finished with value: 10258.960665288203 and parameters: {'max_bid': 222.33659876801272, 'gamma': 0.907135566221697, 'N_bound': 15, 'B_bound': 1353}. Best is trial 0 with value: 10258.960665288203.


CPC_REL: 893.5662331077964, rmse: 2.2294264173375677, SCR: 10258.960665288203


...

CPC_REL: 1005.8659289390461, rmse: 1.2232524844249337, SCR: 14901.769053172924


[I 2026-04-28 12:36:15,726] Trial 23 finished with value: 14917.159571752743 and parameters: {'max_bid': 21.045797839519945, 'gamma': 0.8818117893722377, 'N_bound': 28, 'B_bound': 3927}. Best is trial 23 with value: 14917.159571752743.


CPC_REL: 982.7689988536935, rmse: 1.2449463643987533, SCR: 14917.159571752743


[I 2026-04-28 12:37:09,855] Trial 24 finished with value: 14883.890533863036 and parameters: {'max_bid': 20.32871520047536, 'gamma': 0.934574916315144, 'N_bound': 29, 'B_bound': 4823}. Best is trial 23 with value: 14917.159571752743.


CPC_REL: 982.7706105481464, rmse: 1.244390846956199, SCR: 14883.890533863036
Best trial:
  Value: 14917.159571752743
  Params: 
    max_bid: 21.045797839519945
    gamma: 0.8818117893722377
    N_bound: 28
    B_bound: 3927


In [11]:
# best_params_path = f'best_params/{best_params_subfolder}/{metric.lower()}.pkl'
best_params_path='best_params/rlb_dp_scr_FPA.pkl'
best_model_path = f'best_models/{best_models_subfolder}/{metric.lower()}.pkl'

rlb_bidder_best = train_best_rlb_dp(best_params_path, best_model_path)

Hours: 100%|██████████| 28/28 [00:00<00:00, 102.00it/s]


In [12]:
best_params_path

'best_params/rlb_dp_scr_FPA.pkl'

In [13]:
best_params_rlb = pd.read_pickle(best_params_path)
best_model_rlb_path = best_model_path

In [14]:
campaigns_path_test = data_config['test']['campaigns_path']
stats_path_test = data_config['test']['stats_path']

In [15]:
res = autobidder_check(
    bidder=RLBDPBidder,
    params = {
        "input_campaigns": campaigns_path_test,
        "input_stats": stats_path_test,
        "model_path": best_model_path,
        **best_params_rlb
    },
    auction_mode=auction_mode,
)

In [16]:
print(f"CPC_REL: {res['score'][0]}, rmse: {res['score'][1]}, SCR: {res['score'][2]}")

CPC_REL: 1044.7359370385877, rmse: 1.2820161404703379, SCR: 15081.728715934123
